In [1]:
! pip install torch torchvision pandas numpy matplotlib

In [2]:
import torch
props = torch.cuda.get_device_properties(0)
print(f"Device: {props.name}")
print(f"SM Count: {props.multi_processor_count}")

Device: NVIDIA GeForce RTX 4060 Laptop GPU
SM Count: 24


In [ ]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# ==========================================
# 1. CONFIGURATION
# ==========================================

DATASET_ROOT = './signal_dataset/',
# DATASET_ROOT = './signal_dataset/', # --- final boss dataset

CONFIG = {
    'data_root': DATASET_ROOT,
    'single_csv': f'{DATASET_ROOT}/single/train/metadata/global_metadata.csv',
    'dual_csv': f'{DATASET_ROOT}/dual/train/metadata/global_metadata.csv',
    'batch_size': 64,
    'epochs': 75,
    'lr': 1e-3,
    'weight_decay': 0.05,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'num_classes': 16,
    'variant': 'ronto',
    'log_interval': 10,

    # --- RESUME CONFIGURATION ---
    # Put the path to your .pth file here.
    # If the file exists, it will load it. If not, it starts from scratch.
    'resume_path': './convnext_ronto_latest.pth'
}

MODULATION_CLASSES = [
    'NM', 'LFM', 'DLFM', 'MLFM', 'EQFM',
    'SFM', 'BFSK', 'QFSK', 'BPSK', 'Frank',
    'P1', 'P2', 'P3', 'P4', 'LFM-BPSK'
]
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(MODULATION_CLASSES)}

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def inspect_model(model):
    print("\n" + "="*60)
    print(f"MODEL ARCHITECTURE REPORT ({CONFIG['variant']})")
    print("="*60)
    total_params = 0
    trainable_params = 0
    print(f"{'Layer Name':<40} | {'Shape':<20} | {'Params':<10}")
    print("-" * 75)
    for name, param in model.named_parameters():
        if param.requires_grad:
            num_params = param.numel()
            total_params += num_params
            trainable_params += num_params
            print(f"{name:<40} | {str(list(param.shape)):<20} | {num_params:<10}")
    print("-" * 75)
    print(f"Total Trainable Parameters: {trainable_params:,}")
    print("="*60 + "\n")


def plot_history(history):
    if len(history['train_loss']) == 0: return
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'b-', label='Training Loss')
    plt.plot(epochs, history['val_loss'], 'r--', label='Validation Loss')
    plt.title('Training & Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['val_acc'], 'g-', label='Validation Exact Acc')
    plt.title('Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('training_plots.png')
    print("\n[INFO] Plots saved to 'training_plots.png'")

# ==========================================
# 3. PREPROCESSING & MODEL
# ==========================================
class SignalToSpectrogram:
    def __init__(self, n_fft=512, win_length=101, hop_length=8):
        self.n_fft = n_fft
        self.win_length = win_length
        self.hop_length = hop_length
        self.window = torch.hann_window(win_length)

    def __call__(self, signal):
        if not isinstance(signal, torch.Tensor):
            signal = torch.from_numpy(signal).float()
        stft = torch.stft(
            signal, n_fft=self.n_fft, hop_length=self.hop_length,
            win_length=self.win_length, window=self.window,
            return_complex=True, center=True
        )
        spectrogram = torch.abs(stft).pow(2)
        spectrogram = spectrogram[:256, :256]
        if spectrogram.shape[1] < 256:
            pad = 256 - spectrogram.shape[1]
            spectrogram = torch.nn.functional.pad(spectrogram, (0, pad))
        elif spectrogram.shape[1] > 256:
            spectrogram = spectrogram[:, :256]
        max_val = spectrogram.max()
        if max_val > 0:
            spectrogram = spectrogram / max_val
        return spectrogram.unsqueeze(0)

class LayerNorm2d(nn.Module):
    def __init__(self, num_channels, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(num_channels))
        self.bias = nn.Parameter(torch.zeros(num_channels))
        self.eps = eps
    def forward(self, x):
        u = x.mean(1, keepdim=True)
        s = (x - u).pow(2).mean(1, keepdim=True)
        x = (x - u) / torch.sqrt(s + self.eps)
        x = self.weight[:, None, None] * x + self.bias[:, None, None]
        return x

class ConvNeXtBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm = LayerNorm2d(dim)
        self.pwconv1 = nn.Conv2d(dim, 4 * dim, kernel_size=1)
        self.act = nn.GELU()
        self.pwconv2 = nn.Conv2d(4 * dim, dim, kernel_size=1)
    def forward(self, x):
        input = x
        x = self.dwconv(x)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        x = input + x
        return x

class ConvNeXtSignalClassifier(nn.Module):
    def __init__(self, variant="ronto", num_classes=16, in_chans=1):
        super().__init__()
        dims = [16, 32, 64, 128] if variant == "ronto" else [32, 48, 64, 96]
        self.downsample_layers = nn.ModuleList()
        self.downsample_layers.append(nn.Sequential(
            nn.Conv2d(in_chans, dims[0], kernel_size=4, stride=4),
            LayerNorm2d(dims[0])
        ))
        for i in range(3):
            self.downsample_layers.append(nn.Sequential(
                LayerNorm2d(dims[i]),
                nn.Conv2d(dims[i], dims[i+1], kernel_size=2, stride=2)
            ))
        self.stages = nn.ModuleList()
        for i in range(4):
            self.stages.append(nn.Sequential(ConvNeXtBlock(dim=dims[i])))
        self.norm = nn.LayerNorm(dims[-1], eps=1e-6)
        self.head = nn.Linear(dims[-1], num_classes)
    def forward(self, x):
        for i in range(4):
            x = self.downsample_layers[i](x)
            x = self.stages[i](x)
        x = self.norm(x.mean([-2, -1]))
        return self.head(x)

# ==========================================
# 4. DATASET
# ==========================================
class FilePathSignalDataset(Dataset):
    def __init__(self, single_csv, dual_csv, data_root, transform=None):
        self.transform = transform
        self.data_root = data_root
        self.preprocessor = SignalToSpectrogram()
        self.load_failures = 0
        self.load_successes = 0

        print(f"Reading Metadata...")
        df_s = pd.read_csv(single_csv)
        if 'modulation1' not in df_s.columns: df_s['modulation1'] = None
        if 'modulation2' not in df_s.columns: df_s['modulation2'] = None
        if 'snr_db' not in df_s.columns: df_s['snr_db'] = 0
        df_s['is_dual_simulated'] = 0.0

        df_d = pd.read_csv(dual_csv)
        if 'modulation' not in df_d.columns: df_d['modulation'] = None
        if 'snr_db' not in df_d.columns: df_d['snr_db'] = 0
        df_d['is_dual_simulated'] = 1.0

        cols = ['signal_path', 'modulation', 'modulation1', 'modulation2', 'is_dual_simulated', 'snr_db']
        cols_s = [c for c in cols if c in df_s.columns]
        cols_d = [c for c in cols if c in df_d.columns]

        self.data = pd.concat([df_s[cols_s], df_d[cols_d]], ignore_index=True)
        self.data = self.data.sample(frac=1).reset_index(drop=True)
        print(f"-> Metadata Loaded. Total Samples: {len(self.data)}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        rel_path = str(row['signal_path'])
        full_path = os.path.join(self.data_root, rel_path)

        snr = row.get('snr_db', 0)

        try:
            signal = np.load(full_path).flatten()
            self.load_successes += 1
        except Exception:
            signal = np.zeros(2048, dtype=np.float32)
            self.load_failures += 1
            if self.load_failures < 5:
                print(f"[WARN] Failed to load: {full_path}")

        if len(signal) > 2048: signal = signal[:2048]
        elif len(signal) < 2048: signal = np.pad(signal, (0, 2048 - len(signal)))
        signal = signal.astype(np.float32)

        spectrogram = self.preprocessor(signal)
        if self.transform: spectrogram = self.transform(spectrogram)

        target = torch.zeros(16, dtype=torch.float32)
        if pd.notna(row.get('modulation')) and row['modulation'] in CLASS_TO_IDX:
            target[CLASS_TO_IDX[row['modulation']]] = 1.0
        if pd.notna(row.get('modulation1')) and row['modulation1'] in CLASS_TO_IDX:
            target[CLASS_TO_IDX[row['modulation1']]] = 1.0
        if pd.notna(row.get('modulation2')) and row['modulation2'] in CLASS_TO_IDX:
            target[CLASS_TO_IDX[row['modulation2']]] = 1.0
        target[15] = row['is_dual_simulated']

        return spectrogram, target, snr

# ==========================================
# 5. TRAINING LOOP (RESUMABLE)
# ==========================================
def main():
    train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomResizedCrop(size=(256, 256), scale=(0.08, 1.0), ratio=(0.75, 1.25)),
        transforms.RandomErasing(p=0.33, scale=(0.02, 0.33))
    ])

    print("\n--- 1. DATASET INITIALIZATION ---")
    try:
        full_dataset = FilePathSignalDataset(
            CONFIG['single_csv'], CONFIG['dual_csv'], CONFIG['data_root'], transform=train_transform
        )

        train_size = int(0.8 * len(full_dataset))
        val_size = len(full_dataset) - train_size
        train_ds, val_ds = torch.utils.data.random_split(full_dataset, [train_size, val_size])
        val_ds.dataset.transform = None

        print(f"-> Train Set: {train_size} samples")
        print(f"-> Val Set:   {val_size} samples")

        train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=0)
        val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0)

    except Exception as e:
        print(f"[FATAL] Dataset Error: {e}")
        return

    print("\n--- 2. MODEL INITIALIZATION ---")
    model = ConvNeXtSignalClassifier(variant=CONFIG['variant']).to(CONFIG['device'])
    inspect_model(model)

    optimizer = optim.AdamW(model.parameters(), lr=CONFIG['lr'], betas=(0.9, 0.995), weight_decay=CONFIG['weight_decay'])
    criterion = nn.BCEWithLogitsLoss()

    # --- CHECKPOINT LOADING LOGIC ---
    start_epoch = 0
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    if os.path.exists(CONFIG['resume_path']):
        print(f"\n[RESUME] Found checkpoint: {CONFIG['resume_path']}")
        checkpoint = torch.load(CONFIG['resume_path'], map_location=CONFIG['device'])

        # Scenario A: Full Checkpoint (Created by this new script)
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            start_epoch = checkpoint['epoch']
            history = checkpoint.get('history', history)
            print(f"-> Resuming from Epoch {start_epoch}")

        # Scenario B: Weights Only (Old style save)
        else:
            model.load_state_dict(checkpoint)
            print("-> Loaded weights only. Starting from Epoch 0 (Optimizer reset).")
    else:
        print("\n[INIT] No checkpoint found. Starting from scratch.")

    start_time = time.time()

    print("\n--- 3. STARTING TRAINING ---")
    for epoch in range(start_epoch, CONFIG['epochs']):
        # --- TRAINING ---
        model.train()
        running_loss = 0.0

        print(f"\n[Epoch {epoch+1}/{CONFIG['epochs']}] Training...")
        for batch_idx, (inputs, targets, snrs) in enumerate(train_loader):
            inputs, targets = inputs.to(CONFIG['device']), targets.to(CONFIG['device'])

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            if torch.isnan(loss) or torch.isinf(loss):
                print(f"[CRITICAL] Loss exploded to {loss.item()} at batch {batch_idx}! Stopping.")
                return

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

            if batch_idx % CONFIG['log_interval'] == 0:
                print(f"  > [Train] Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

        epoch_loss = running_loss / len(train_ds)
        history['train_loss'].append(epoch_loss)

        # --- VALIDATION ---
        model.eval()
        val_loss = 0.0
        correct_exact = 0
        total_samples = 0
        snr_stats = {}

        print(f"[Epoch {epoch+1}/{CONFIG['epochs']}] Validating...")

        with torch.no_grad():
            for batch_idx, (inputs, targets, snrs) in enumerate(val_loader):
                inputs, targets = inputs.to(CONFIG['device']), targets.to(CONFIG['device'])
                outputs = model(inputs)
                v_loss = criterion(outputs, targets)
                val_loss += v_loss.item() * inputs.size(0)

                preds = (torch.sigmoid(outputs) > 0.5).float()
                matches = (preds == targets).all(dim=1)
                correct_exact += matches.sum().item()
                total_samples += inputs.size(0)

                for i in range(len(snrs)):
                    cur_snr = int(snrs[i].item())
                    is_correct = matches[i].item()
                    if cur_snr not in snr_stats: snr_stats[cur_snr] = {'correct': 0, 'total': 0}
                    snr_stats[cur_snr]['total'] += 1
                    if is_correct: snr_stats[cur_snr]['correct'] += 1

                if batch_idx % CONFIG['log_interval'] == 0:
                    current_acc = (correct_exact / total_samples) * 100
                    print(f"  > [Val]   Batch {batch_idx}/{len(val_loader)} | Loss: {v_loss.item():.4f} | Running Acc: {current_acc:.2f}%")

        val_epoch_loss = val_loss / len(val_ds)
        val_acc = correct_exact / total_samples
        history['val_loss'].append(val_epoch_loss)
        history['val_acc'].append(val_acc)

        # --- SUMMARY ---
        elapsed = time.time() - start_time
        print(f"\n--- Epoch {epoch+1} Summary ---")
        print(f"  Time Elapsed : {elapsed/60:.2f} mins")
        print(f"  Train Loss   : {epoch_loss:.5f}")
        print(f"  Val Loss     : {val_epoch_loss:.5f}")
        print(f"  Val Exact Acc: {val_acc*100:.2f}%")

        # --- SNR TABLE ---
        print("\n  --- SNR Breakdown (Accuracy) ---")
        for s in sorted(snr_stats.keys()):
            stats = snr_stats[s]
            acc = (stats['correct'] / stats['total']) * 100 if stats['total'] > 0 else 0
            print(f"  SNR {s:>3} dB : {acc:6.2f}%  (Count: {stats['total']})")
        print("  --------------------------------")

        if full_dataset.load_failures > 0:
            fail_rate = full_dataset.load_failures / (full_dataset.load_failures + full_dataset.load_successes)
            if fail_rate > 0.1:
                print(f"  [WARN] High Data Load Failure Rate: {fail_rate*100:.1f}% files missing!")

        plot_history(history)

        # --- ROBUST SAVING (Save Dict instead of just weights) ---
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'history': history
        }
        torch.save(checkpoint, CONFIG['resume_path'])
        # Also save a backup just in case
        torch.save(model.state_dict(), "weights_only_backup.pth")

    print("\nTraining Complete. Final model saved.")

if __name__ == "__main__":
    main()


--- 1. DATASET INITIALIZATION ---
Reading Metadata...
-> Metadata Loaded. Total Samples: 957000
-> Train Set: 765600 samples
-> Val Set:   191400 samples

--- 2. MODEL INITIALIZATION ---

MODEL ARCHITECTURE REPORT (ronto)
Layer Name                               | Shape                | Params    
---------------------------------------------------------------------------
downsample_layers.0.0.weight             | [16, 1, 4, 4]        | 256       
downsample_layers.0.0.bias               | [16]                 | 16        
downsample_layers.0.1.weight             | [16]                 | 16        
downsample_layers.0.1.bias               | [16]                 | 16        
downsample_layers.1.0.weight             | [16]                 | 16        
downsample_layers.1.0.bias               | [16]                 | 16        
downsample_layers.1.1.weight             | [32, 16, 2, 2]       | 2048      
downsample_layers.1.1.bias               | [32]                 | 32        
downsamp

KeyboardInterrupt: 